# 06.1 · Exercises

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cloudops-one/training-ai-ml/blob/main/modules/06-unsupervised/01-what-is-unsupervised/exercises.ipynb)

Run the setup cells, then try each exercise before opening [solutions.ipynb](solutions.ipynb).

In [ ]:
# --- data helper: works locally and on Colab -------------------------------
from pathlib import Path
import urllib.request

def data_path(name):
    """Return a local path to data/<name>, downloading from GitHub on Colab."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidate = parent / "data" / name
        if candidate.exists():
            return candidate
    url = f"https://raw.githubusercontent.com/cloudops-one/training-ai-ml/main/data/{name}"
    dest = Path(name)
    if not dest.exists():
        urllib.request.urlretrieve(url, dest)
    return dest

import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings("ignore")

In [ ]:
import sys
sys.path.insert(0, str(data_path("download.py").parent))
from download import load_retail

raw = load_retail("Year 2010-2011")
sales = raw[~raw["Invoice"].astype(str).str.startswith("C")]
sales = sales[(sales["Quantity"] > 0) & (sales["Price"] > 0)].dropna(subset=["Customer ID"]).drop_duplicates()
sales = sales.assign(Revenue=sales["Quantity"] * sales["Price"])

snapshot = sales["InvoiceDate"].max().normalize() + pd.Timedelta(days=1)    # 2011-12-10
rfm = sales.groupby("Customer ID").agg(
    recency=("InvoiceDate", lambda d: (snapshot - d.max()).days),
    frequency=("Invoice", "nunique"),
    monetary=("Revenue", "sum"))
print(f"{len(sales):,} rows, {len(rfm):,} customers, snapshot {snapshot.date()}")

In [ ]:
from sklearn.preprocessing import StandardScaler

rfm_log = np.log1p(rfm)                         # tame the long right tails
X = StandardScaler().fit_transform(rfm_log)      # mean 0, std 1 per column
print("X shape:", X.shape)

## Exercise 1: three, four, five groups

Run K-Means with `n_clusters` = 3, 4 and 5 (`n_init=10, random_state=42`). For each, print the number of customers and the mean R, F, M per group, sorted by monetary. Which group from the 4-group version splits when you go to 5? Which groups merge when you go to 3?

In [ ]:
# your code here

## Exercise 2: clustering pure noise

Make 1,000 rows of random numbers with two columns (`np.random.default_rng(0).normal(size=(1000, 2))`). Run K-Means with 4 clusters and plot the points coloured by cluster. Does K-Means complain that there is no structure?

In [ ]:
# your code here

## Exercise 3: more pairs

Rebuild the pair counts from the lesson and print the top 10 pairs. Pick two you would bundle and one you would not, and say why.

In [ ]:
# your code here